**Lifetime Value (LTV) Project**

This notebook focuses on forecasting Customer Lifetime Value (LTV) using historical transaction data. The dataset is transformed into time series sequences to capture temporal patterns and customer behavior over time.

Two forecasting approaches are explored:

*   Long Short-Term Memory (LSTM), a recurrent neural network designed for sequential data,
*   TimeFM, a foundation model for time series forecasting.

Their performance is evaluated to assess their effectiveness in projecting future customer value.

## Setup and Dependencies

### Library

Imports the libraries and dependencies required throughout the project.

Install the TimesFM foundation model library (PyTorch backend).

In [ ]:
!pip install -q timesfm[torch]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import auth
from google.colab import drive

import timesfm

from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

### Auth

Authentication to google account and drive for more google cloud services access

In [ ]:
drive.mount('/content/drive')

auth.authenticate_user()

## Data Preparation

### Loader

Query for access data in Google BigQuery

Load data from bigquery

In [ ]:
df = pd.read_gbq(sql_query, project_id='', dialect='standard')
df.head()

In [ ]:
df.tail()

### Exploration

Compact information about data

In [ ]:
df.info()

Checking unique value

In [ ]:
unique_values = df.nunique()
print(unique_values)

Checking cohort list

In [ ]:
sorted(df['date_created_at'].unique())

Checking duplicate data

In [ ]:
df.duplicated().sum()

### Feature Engineering

Adding cohort age (date update - date created) for getting information day every cohort

In [ ]:
df['cohort_age'] = (
    df['date_update'] -
    df['date_created_at']
).dt.days

In [ ]:
df.head(10)

Adding day of week using pandas format (Monday is starting from 0)

In [ ]:
df['date_created_at'] = pd.to_datetime(df['date_created_at'])
df['date_update'] = pd.to_datetime(df['date_update'])

df['day_of_week'] = df['date_update'].dt.dayofweek

In [ ]:
df.head()

Checking data information

In [ ]:
df.info()

### Visualization

Distribution of cohort length

In [ ]:
cohort_length = (
    df.groupby('date_created_at')['cohort_age']
      .max()
      .add(1)
      .reset_index(name='cohort_length')
)

In [ ]:
dist = (
    cohort_length['cohort_length']
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(10, 5))

bars = plt.bar(
    dist.index.astype(str),
    dist.values
)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        str(int(height)),
        ha='center',
        va='bottom'
    )

plt.xlabel('Cohort Length (days)')
plt.ylabel('Number of Cohorts')
plt.title('Distribution of Cohort Length')
plt.grid(axis='y', alpha=0.3)

plt.show()

Distribution of running revenue per cohort

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample = (
    df['date_created_at']
    .drop_duplicates()
    .sample(30)
)

plt.figure(figsize=(14, 8))

for cohort in sorted(sample):
    temp = (
        df[df['date_created_at'] == cohort]
        .sort_values('cohort_age')
    )

    plt.plot(
        temp['cohort_age'],
        temp['revenue_running_total'],
        linewidth=2,
        alpha=0.8,
        label=cohort.strftime('%Y-%m-%d')
    )

plt.title("Running Revenue per Cohort", fontsize=16)
plt.xlabel("Cohort Age (Days)")
plt.ylabel("Running Revenue")
plt.grid(alpha=0.3)

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    borderaxespad=0
)

plt.tight_layout()
plt.show()

Average revenue daily push distribution per cohort age

In [ ]:
avg_daily = (
    df.groupby('cohort_age')['revenue_dailypush']
    .mean()
)

avg_daily.plot()

Average total revenue distribution per cohort age

In [ ]:
avg_running = (
    df.groupby('cohort_age')['revenue_running_total']
    .mean()
)

avg_running.plot()

Numerical correlation

In [ ]:
numeric_cols = [
    'cohort_age',
    'no_of_user',
    'revenue_dailypush',
    'revenue_running_total'
]

corr = df[numeric_cols].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm'
)

## Preprocessing

### Sort Data

Final preparation with sorting date created and cohort age

In [ ]:
df = (
    df
    .sort_values(
        ['date_created_at', 'cohort_age']
    )
    .reset_index(drop=True)
)

In [ ]:
df.head()

### Encoding

Create new dataframe for preprocessing process

In [ ]:
df_model = df.copy()

subject_encoder = LabelEncoder()
adnet_encoder = LabelEncoder()

Subject encoding

In [ ]:
df_model['subject_encoded'] = subject_encoder.fit_transform(df_model['subject'])

In [ ]:
df_model.head()

Adnet encoding

In [ ]:
df_model['adnet_encoded'] = adnet_encoder.fit_transform(df_model['adnet'])

In [ ]:
df_model.head()

### Feature Selection

Selecting feature for new dataframe

In [ ]:
KEEP_COLS = [
    'date_created_at',
    'cohort_age',
    'revenue_running_total',
    'revenue_dailypush',
    'no_of_user',
    'day_of_week',
    'subject_encoded',
    'adnet_encoded'
]

df_model = df_model[KEEP_COLS]

In [ ]:
df_model.head()

### Split Data

Split to train 80% and validation 20% from total cohorts

In [ ]:
cohorts = sorted(df_model['date_created_at'].unique())

split_idx = int(len(cohorts) * 0.8)

train_cohorts = cohorts[:split_idx]
val_cohorts = cohorts[split_idx:]

In [ ]:
print("Total Train Cohort:", len(train_cohorts))
print("Sample Train Cohort:", train_cohorts)

print("Total Val Cohort:", len(val_cohorts))
print("Sample Val Cohort:", val_cohorts)

Insert to pandas for next step (forecasting)

In [ ]:
train_df = df_model[df_model['date_created_at'].isin(train_cohorts)]
val_df = df_model[df_model['date_created_at'].isin(val_cohorts)]

In [ ]:
train_df.head()

### Target

Setup feature and target for training

In [ ]:
FEATURES = [
    'revenue_running_total',
    'cohort_age',
    'day_of_week',
]

TARGET = 'revenue_running_total'

TimesFM is a univariate foundation model that performs its own input normalization internally. Manual Min-Max scaling is therefore not applied here — the model consumes and forecasts `revenue_running_total` directly in its original scale, using only the target series (the other features are kept for reference).

Checking data train

In [ ]:
train_df.info()

In [ ]:
train_df.head()

Checking data validation

In [ ]:
val_df.info()

In [ ]:
val_df.head()

## Forecast Setup

### Series Builder

TimesFM forecasts a whole horizon from a contiguous context window, so instead of sliding windows we simply extract the ordered target series for each cohort. The configuration below sets the model context/horizon limits and the daily frequency indicator.

In [ ]:
CONTEXT_LEN = 2048   # max context length the model can attend to
HORIZON_LEN = 512    # max forecast horizon per call
CONTEXT_MIN = 30     # minimum history required before forecasting
FREQ        = 0      # 0 = high-frequency (daily) series

def get_series(data, cohort):
    """Return the ordered univariate target series for one cohort."""
    s = (
        data[data['date_created_at'] == cohort]
        .sort_values('cohort_age')
    )
    return s[TARGET].values.astype(float)

## Modeling

### TimesFM

TimesFM is a pretrained foundation model, so it is used zero-shot — there is no training loop. Load the 2.0 500M PyTorch checkpoint with the fixed architecture parameters.

In [ ]:
tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend="gpu",            # change to "cpu" if no GPU is available
        per_core_batch_size=32,
        horizon_len=HORIZON_LEN,
        context_len=CONTEXT_LEN,
        input_patch_len=32,
        output_patch_len=128,
        num_layers=50,
        model_dims=1280,
        use_positional_embedding=False,
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id="google/timesfm-2.0-500m-pytorch"
    ),
)

## Evaluation

### Visualization

Evaluate zero-shot on the validation cohorts: for each cohort use all but the last `EVAL_HORIZON` days as context and forecast the held-out tail, then compare against the actuals.

In [ ]:
EVAL_HORIZON = 30

eval_contexts = []
eval_actuals  = []

for c in val_cohorts:
    s = get_series(val_df, c)
    if len(s) < CONTEXT_MIN + EVAL_HORIZON:
        continue
    eval_contexts.append(s[:-EVAL_HORIZON])
    eval_actuals.append(s[-EVAL_HORIZON:])

print("Evaluated cohorts:", len(eval_contexts))

point_forecast, _ = tfm.forecast(eval_contexts, freq=[FREQ] * len(eval_contexts))

y_pred = np.concatenate([point_forecast[i, :EVAL_HORIZON] for i in range(len(eval_contexts))])
y_val  = np.concatenate(eval_actuals)

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(
    y_val[:200],
    label='Actual'
)

plt.plot(
    y_pred[:200],
    label='Prediction'
)

plt.title('Validation Prediction')

plt.legend()

plt.show()

In [ ]:
residuals = y_val - y_pred

plt.figure(figsize=(10,5))

plt.hist(
    residuals,
    bins=50
)

plt.title('Residual Distribution')

plt.show()

### Metrics

Detail evaluation metrics in MAE, RMSE, MAPE

In [ ]:
mae = mean_absolute_error(
    y_val,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_val,
        y_pred
    )
)

mape = mean_absolute_percentage_error(
    y_val,
    y_pred
) * 100

In [ ]:
print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"MAPE : {mape:.2f}%")

## Projection

Use the first `SEED_DAYS` of a cohort as context and let TimesFM forecast all remaining days in a single multi-horizon call (no recursive stepping needed).

Select cohort and run projection

In [ ]:
SEED_DAYS = 90   # first 90 days used as context seed

candidates = [
    c for c in val_cohorts
    if int(df[df['date_created_at'] == c]['cohort_age'].max()) >= SEED_DAYS
]

print("Eligible cohorts:", len(candidates))

cohort = candidates[0]

cohort_raw = (
    df[df['date_created_at'] == cohort]
    .sort_values('cohort_age')
    .reset_index(drop=True)
)

seed_mask     = cohort_raw['cohort_age'] < SEED_DAYS
context       = cohort_raw.loc[seed_mask, TARGET].values.astype(float)
actual_future = cohort_raw.loc[~seed_mask, TARGET].values.astype(float)
n_steps       = len(actual_future)

print("Cohort      :", cohort)
print("Seed days   :", len(context))
print("Future days :", n_steps)

point_forecast, _ = tfm.forecast([context], freq=[FREQ])
pred_future = point_forecast[0, :n_steps]

print(f"Predicted running total at day {SEED_DAYS + n_steps - 1}: {pred_future[-1]:,.0f}")
print(f"Actual    running total at day {SEED_DAYS + n_steps - 1}: {actual_future[-1]:,.0f}")

Evaluation at day+30, day+60, and full horizon

In [ ]:
milestones = {'day+30': 30, 'day+60': 60, 'full': n_steps}

for label, days in milestones.items():
    n = min(days, len(actual_future), len(pred_future))
    if n == 0:
        print(f"{label}: not enough data")
        continue
    mae  = mean_absolute_error(actual_future[:n], pred_future[:n])
    rmse = np.sqrt(mean_squared_error(actual_future[:n], pred_future[:n]))
    mape = mean_absolute_percentage_error(actual_future[:n], pred_future[:n]) * 100
    print(f"[{label:>7}] MAE={mae:>12,.0f}  RMSE={rmse:>12,.0f}  MAPE={mape:.2f}%")

Actual vs projection visualization

In [ ]:
seed_age   = cohort_raw.loc[seed_mask, 'cohort_age'].values
future_age = cohort_raw.loc[~seed_mask, 'cohort_age'].values[:n_steps]
pred_age   = future_age[:len(pred_future)]

plt.figure(figsize=(15, 5))

plt.plot(seed_age,   context,       label='Actual (seed 0-89)',   color='steelblue')
plt.plot(future_age, actual_future, label='Actual (future)',       color='green')
plt.plot(pred_age,   pred_future,   label='Projection (TimesFM)',  color='tomato', linestyle='--')

for offset, color in [(30, 'orange'), (60, 'purple')]:
    day = SEED_DAYS + offset - 1
    if len(future_age) > 0 and day <= future_age[-1]:
        plt.axvline(x=day, color=color, linestyle=':', alpha=0.7, label=f'day+{offset}')

plt.axvline(x=SEED_DAYS, color='gray', linestyle=':', alpha=0.7, label='seed end')
plt.title(f'LTV Projection (TimesFM) \u2014 Cohort {str(cohort)[:10]}')
plt.xlabel('Cohort Age (Days)')
plt.ylabel('Running Revenue')
plt.legend()
plt.grid(alpha=0.3)

plt.show()